## Visualize Inliers and Outliers

# Introduction: Seeing the Good and the Bad

Welcome to the second unit of our course. In the previous unit, you learned how to align images by calculating a homography matrix. You practiced extracting coordinates and using the RANSAC algorithm to find an accurate alignment while ignoring incorrect matches.

RANSAC does a fantastic job of filtering out bad matches, but as developers, we need to inspect what it accepted and what it rejected to ensure our alignment is actually sound. A homography might compute mathematically but practically represent a bad alignment.

In this lesson, we will learn how to extract the **inliers** (the good matches RANSAC kept) and the **outliers** (the bad matches it threw away) into separate lists. We will write a diagnostic tool to evaluate the alignment quality automatically, and then we will draw the results on the screen to visually confirm our math.

---

## Extracting Inliers and Outliers

As a quick reminder, when we estimate a homography using RANSAC, the algorithm returns a transformation matrix and an inlier mask. This mask is essentially a list of 1s and 0s telling us exactly which matches fit the estimated geometry (`1`) and which do not (`0`).

To make use of this mask, we need to split our original list of matches into two separate lists. We can achieve this cleanly in Python using the `zip()` function, which lets us iterate through the matches and the mask simultaneously.

Let's look at how we can use list comprehensions to separate our data:

```python
# Assuming 'matches' is our list of features and 'inliers' is our mask of 1s and 0s
inlier_matches = [m for m, keep in zip(matches, inliers) if keep]

```

In this code, `zip(matches, inliers)` pairs each match object `m` with its corresponding boolean/binary value, which we call `keep`. If `keep` is `1` (which evaluates to `True` in Python), the match is added to our `inlier_matches` list.

We can apply the exact same logic to find the outliers by simply looking for the items we do not want to keep:

```python
inlier_matches = [m for m, keep in zip(matches, inliers) if keep]
outlier_matches = [m for m, keep in zip(matches, inliers) if not keep]

print("Matches:", len(matches))
print("Inliers:", len(inlier_matches))
print("Outliers:", len(outlier_matches))

```

**Output:**

```text
Matches: 450
Inliers: 120
Outliers: 330

```

By separating them, we now have precise control over our data for both quantitative analysis and visual debugging.

---

## Diagnosing the Alignment

Just looking at the raw count of inliers and outliers is not always enough. We want our program to automatically interpret the match data and warn us if something looks wrong. To do this, we can build a `diagnose_alignment` function.

First, we calculate key statistics from our `inliers` boolean mask:

* **Total inliers:** `inliers.sum()`
* **Inlier ratio:** `inliers.mean()` (with a safeguard against division by zero)

```python
def diagnose_alignment(match_count, inliers):
    inlier_count = int(inliers.sum())
    inlier_ratio = float(inliers.mean()) if len(inliers) else 0.0

```

Now that we have our metrics, we can add a series of diagnostic checks. Each scenario reflects a specific practical failure mode:

```python
def diagnose_alignment(match_count, inliers):
    inlier_count = int(inliers.sum())
    inlier_ratio = float(inliers.mean()) if len(inliers) else 0.0

    if match_count < 20:
        return "diagnosis: too few matches; revisit the matching report"
    if inlier_count < 10:
        return "diagnosis: too few inliers; the images may not have enough reliable overlap"
    if inlier_ratio < 0.25:
        return "diagnosis: many matches disagree geometrically; try a stricter ratio or a better image pair"

    return "diagnosis: alignment is plausible; inspect the inlier visualization"

```

### Breakdown of Alignment Checks:

* **Too few matches overall ($< 20$):** Indicates a failure in the initial feature detection or matching stage before RANSAC runs.
* **Too few inliers ($< 10$):** Indicates insufficient geometric overlap between views to reliably compute a homography.
* **Low inlier ratio ($< 0.25$):** A large volume of matches with few inliers (e.g., 10%) points to repetitive patterns (such as brick walls), moving objects, or severe parallax.

Running mock data through the diagnostic function produces clear, actionable feedback:

```python
# Assuming 450 total matches and 'inliers' boolean mask
print("inlier ratio:", float(inliers.mean()))
print(diagnose_alignment(450, inliers))

```

**Output:**

```text
inlier ratio: 0.266
diagnosis: alignment is plausible; inspect the inlier visualization

```

---

## Visualizing the Results

Numbers and text provide quick verification, but visual inspection remains essential for assessing alignment accuracy. We can use OpenCV's `cv2.drawMatches` to render correspondence lines between views.

### Drawing Inliers (Green)

```python
import cv2

inlier_view = cv2.drawMatches(
    left_image,
    keypoints1,
    right_image,
    keypoints2,
    inlier_matches[:80],
    None,
    matchColor=(0, 255, 0),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

```

**Practical Rendering Tips:**

1. **Match Limiting (`[:80]`):** Slicing the match list prevents visual clutter and unreadable dense line clusters.
2. **`NOT_DRAW_SINGLE_POINTS` Flag:** Ensures OpenCV draws only keypoints that are actively matched, hiding isolated points.

---

### Drawing Outliers (Red)

We follow the same structure to visualize rejected matches, using red `(0, 0, 255)` for the line color:

```python
outlier_view = cv2.drawMatches(
    left_image,
    keypoints1,
    right_image,
    keypoints2,
    outlier_matches[:80],
    None,
    matchColor=(0, 0, 255),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

```

Displaying these views side-by-side using `cv2.imshow()` makes it easy to spot outliers—such as lines crossing toward moving vehicles, swaying trees, or background depth changes caused by parallax.

---

## Summary and Practice

In this lesson, we established a systematic way to evaluate RANSAC performance:

* **Separation:** Split match lists into inliers and outliers using the boolean mask and `zip()`.
* **Automated Diagnosis:** Built a rule-based `diagnose_alignment` function to catch low match counts, insufficient overlap, or low consensus ratios.
* **Inspection:** Rendered clean, color-coded diagnostic visualizations with `cv2.drawMatches` (green for inliers, red for outliers).

You are now ready to implement these functions in the practice exercises and inspect alignment quality directly.

## Diagnosing Alignment Quality from Matches

Now that the lesson has shown how RANSAC separates inliers from outliers, it is time to apply that knowledge to a small diagnostic helper.

Your task is to complete the diagnose_alignment function so that it can read the inlier mask and report a clear verdict regarding the alignment quality.

Follow the TODO comments inside the function to:

    Count the inliers and compute the inlier ratio (handling the empty case).
    Return the "too few matches" message when match_count is below 20.
    Return the "too few inliers" message when the inlier count is below 10.
    Return the "many matches disagree geometrically" message when the ratio is below 0.25.
    Return the "alignment is plausible" message when everything looks healthy.

A robust diagnostic helper will make each subsequent visualization step significantly easier to trust.


```
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def diagnose_alignment(match_count, inliers):
    # TODO: Compute `inlier_count` as an integer from `inliers.sum()`

    # TODO: Compute `inlier_ratio` as a float from `inliers.mean()`,
    #       defaulting to 0.0 if `inliers` is empty

    # TODO: If `match_count` is less than 20, return:
    #       "diagnosis: too few matches; revisit the matching report"

    # TODO: If `inlier_count` is less than 10, return:
    #       "diagnosis: too few inliers; the images may not have enough reliable overlap"

    # TODO: If `inlier_ratio` is less than 0.25, return:
    #       "diagnosis: many matches disagree geometrically; try a stricter ratio or a better image pair"

    # TODO: Otherwise, return:
    #       "diagnosis: alignment is plausible; inspect the inlier visualization"
    pass


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)

    try:
        _, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=args.ransac_threshold,
        )
    except ValueError as exc:
        print("alignment failed:", exc)
        print("diagnosis: revisit overlap, texture, matching method, and ratio threshold")
        return


if __name__ == "__main__":
    main()

```

Here is the completed implementation of `diagnose_alignment` in `solution.py`:

```python
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def diagnose_alignment(match_count, inliers):
    # Compute inlier count
    inlier_count = int(inliers.sum()) if inliers is not None and len(inliers) else 0

    # Compute inlier ratio, defaulting to 0.0 if empty
    inlier_ratio = float(inliers.mean()) if inliers is not None and len(inliers) else 0.0

    # Check 1: Too few matches
    if match_count < 20:
        return "diagnosis: too few matches; revisit the matching report"

    # Check 2: Too few inliers
    if inlier_count < 10:
        return "diagnosis: too few inliers; the images may not have enough reliable overlap"

    # Check 3: Low inlier ratio
    if inlier_ratio < 0.25:
        return "diagnosis: many matches disagree geometrically; try a stricter ratio or a better image pair"

    # Healthy alignment
    return "diagnosis: alignment is plausible; inspect the inlier visualization"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)

    try:
        _, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=args.ransac_threshold,
        )
    except ValueError as exc:
        print("alignment failed:", exc)
        print("diagnosis: revisit overlap, texture, matching method, and ratio threshold")
        return


if __name__ == "__main__":
    main()

```

## Reporting Match Statistics in Main

Nice work building the diagnose_alignment verdict in the previous step — now it is time to put it to use inside main() and turn the raw RANSAC mask into a readable report.

The pipeline already runs feature detection, matching, and homography estimation for you. What is missing is the bookkeeping right after estimate_homography returns its inliers mask.

Follow the TODO comments in main() to:

    Build inlier_matches and outlier_matches by zipping matches with inliers inside a list comprehension.
    Print the total match count, inlier count, and outlier count using the exact prefixes shown in the comments.
    Print the inlier ratio as float(inliers.mean()) with the "inlier ratio:" prefix.
    Call diagnose_alignment(len(matches), inliers) and print whatever it returns.

Once this is in place, you will have a complete textual report that tells you at a glance whether an alignment is worth trusting — the perfect setup for the visualization work coming next.

```
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def diagnose_alignment(match_count, inliers):
    inlier_count = int(inliers.sum())
    inlier_ratio = float(inliers.mean()) if len(inliers) else 0.0

    if match_count < 20:
        return "diagnosis: too few matches; revisit the matching report"

    if inlier_count < 10:
        return "diagnosis: too few inliers; the images may not have enough reliable overlap"

    if inlier_ratio < 0.25:
        return "diagnosis: many matches disagree geometrically; try a stricter ratio or a better image pair"

    return "diagnosis: alignment is plausible; inspect the inlier visualization"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)

    try:
        _, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=args.ransac_threshold,
        )
    except ValueError as exc:
        print("alignment failed:", exc)
        print("diagnosis: revisit overlap, texture, matching method, and ratio threshold")
        return

    # TODO: Build `inlier_matches` by keeping items from `matches` where the
    #       corresponding value in the `inliers` mask is truthy (hint: use a list
    #       comprehension over `zip(matches, inliers)`)

    # TODO: Build `outlier_matches` by keeping items from `matches` where the
    #       corresponding value in the `inliers` mask is NOT truthy

    # TODO: Print the total number of matches, prefixed by "matches:"

    # TODO: Print the number of inliers, prefixed by "inliers:"

    # TODO: Print the number of outliers, prefixed by "outliers:"

    # TODO: Print the inlier ratio as `float(inliers.mean())`, prefixed by "inlier ratio:"

    # TODO: Call `diagnose_alignment(len(matches), inliers)` and print its return value


if __name__ == "__main__":
    main()

```

Berikut adalah kode lengkap `solution.py` dengan implementasi pelaporan statistik kecocokan (*matches*) dan diagnosis di dalam `main()`:

```python
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def diagnose_alignment(match_count, inliers):
    inlier_count = int(inliers.sum())
    inlier_ratio = float(inliers.mean()) if len(inliers) else 0.0

    if match_count < 20:
        return "diagnosis: too few matches; revisit the matching report"

    if inlier_count < 10:
        return "diagnosis: too few inliers; the images may not have enough reliable overlap"

    if inlier_ratio < 0.25:
        return "diagnosis: many matches disagree geometrically; try a stricter ratio or a better image pair"

    return "diagnosis: alignment is plausible; inspect the inlier visualization"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)

    try:
        _, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=args.ransac_threshold,
        )
    except ValueError as exc:
        print("alignment failed:", exc)
        print("diagnosis: revisit overlap, texture, matching method, and ratio threshold")
        return

    # 1. Pisahkan inliers dan outliers menggunakan zip(matches, inliers)
    inlier_matches = [m for m, keep in zip(matches, inliers) if keep]
    outlier_matches = [m for m, keep in zip(matches, inliers) if not keep]

    # 2. Cetak total matches, inliers, dan outliers
    print("matches:", len(matches))
    print("inliers:", len(inlier_matches))
    print("outliers:", len(outlier_matches))

    # 3. Cetak inlier ratio
    print("inlier ratio:", float(inliers.mean()))

    # 4. Cetak hasil diagnosis
    print(diagnose_alignment(len(matches), inliers))


if __name__ == "__main__":
    main()

```

## Showing Inliers and Outliers Side by Side

With the textual report in place, it is time to actually see which matches survived RANSAC and which were thrown out.

In this exercise, you will finish main() by drawing two side-by-side views: one with inlier matches in green, and one with outlier matches in red.

Work through the TODOs near the bottom of main():

    Use cv2.drawMatches to build inlier_view from the first 80 inlier matches, using green ((0, 255, 0)) as the matchColor and cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS for flags.
    Do the same to build outlier_view from the first 80 outlier matches, but use red ((0, 0, 255)).
    Show inlier_view in a window called "inliers" and outlier_view in a window called "outliers" with cv2.imshow.
    Call cv2.waitKey(0) so the windows stay open, then cv2.destroyAllWindows() to clean them up.

Run the program in the terminal, for example: python3 solution.py sample_images/building/1.jpg sample_images/building/2.jpg sample_images/building/3.jpg Once this runs, you'll have a clear visual sense of how RANSAC separated the good matches from the bad ones.

```
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def diagnose_alignment(match_count, inliers):
    inlier_count = int(inliers.sum())
    inlier_ratio = float(inliers.mean()) if len(inliers) else 0.0

    if match_count < 20:
        return "diagnosis: too few matches; revisit the matching report"

    if inlier_count < 10:
        return "diagnosis: too few inliers; the images may not have enough reliable overlap"

    if inlier_ratio < 0.25:
        return "diagnosis: many matches disagree geometrically; try a stricter ratio or a better image pair"

    return "diagnosis: alignment is plausible; inspect the inlier visualization"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)

    try:
        _, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=args.ransac_threshold,
        )
    except ValueError as exc:
        print("alignment failed:", exc)
        print("diagnosis: revisit overlap, texture, matching method, and ratio threshold")
        return

    inlier_matches = [m for m, keep in zip(matches, inliers) if keep]
    outlier_matches = [m for m, keep in zip(matches, inliers) if not keep]

    print("matches:", len(matches))
    print("inliers:", len(inlier_matches))
    print("outliers:", len(outlier_matches))
    print("inlier ratio:", float(inliers.mean()))
    print(diagnose_alignment(len(matches), inliers))

    # TODO: Use `cv2.drawMatches` to build `inlier_view`. Pass `left`, `kp1`, `right`,
    #       `kp2`, `inlier_matches[:80]`, and `None`, then add `matchColor=(0, 255, 0)`
    #       (green because these matches were accepted) and
    #       `flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS`

    # TODO: Use `cv2.drawMatches` the same way to build `outlier_view`, but pass
    #       `outlier_matches[:80]` and `matchColor=(0, 0, 255)` (red for rejected matches)

    # TODO: Show the inlier view in a window named "inliers" using `cv2.imshow`

    # TODO: Show the outlier view in a window named "outliers" using `cv2.imshow`

    # TODO: Call `cv2.waitKey(0)` so the windows stay open until a key is pressed

    # TODO: Call `cv2.destroyAllWindows()` to clean up the windows at the end


if __name__ == "__main__":
    main()

```

Here is the completed `solution.py` file with the visualization steps implemented:

```python
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def diagnose_alignment(match_count, inliers):
    inlier_count = int(inliers.sum())
    inlier_ratio = float(inliers.mean()) if len(inliers) else 0.0

    if match_count < 20:
        return "diagnosis: too few matches; revisit the matching report"

    if inlier_count < 10:
        return "diagnosis: too few inliers; the images may not have enough reliable overlap"

    if inlier_ratio < 0.25:
        return "diagnosis: many matches disagree geometrically; try a stricter ratio or a better image pair"

    return "diagnosis: alignment is plausible; inspect the inlier visualization"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)

    try:
        _, inliers = estimate_homography(
            kp1,
            kp2,
            matches,
            ransac_threshold=args.ransac_threshold,
        )
    except ValueError as exc:
        print("alignment failed:", exc)
        print("diagnosis: revisit overlap, texture, matching method, and ratio threshold")
        return

    inlier_matches = [m for m, keep in zip(matches, inliers) if keep]
    outlier_matches = [m for m, keep in zip(matches, inliers) if not keep]

    print("matches:", len(matches))
    print("inliers:", len(inlier_matches))
    print("outliers:", len(outlier_matches))
    print("inlier ratio:", float(inliers.mean()))
    print(diagnose_alignment(len(matches), inliers))

    # Build the inlier visualization (green lines for accepted matches)
    inlier_view = cv2.drawMatches(
        left,
        kp1,
        right,
        kp2,
        inlier_matches[:80],
        None,
        matchColor=(0, 255, 0),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )

    # Build the outlier visualization (red lines for rejected matches)
    outlier_view = cv2.drawMatches(
        left,
        kp1,
        right,
        kp2,
        outlier_matches[:80],
        None,
        matchColor=(0, 0, 255),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )

    # Show inlier and outlier windows
    cv2.imshow("inliers", inlier_view)
    cv2.imshow("outliers", outlier_view)

    # Wait for key press and clean up windows
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```